# Exercise 6 · LLM-as-a-judge — DEMO SOLUTION
*Lecturer version: one worked solution for section 8. Runs on the 10 example rows (cancel the file picker).*
**ETH MAS 2026 · Module 1 · B8 — Benchmarking & evaluation**

You built a skill in B6. Claude runs it — but it does not tell you whether it is good enough to share with your team.
In this exercise **two models judge your skill's answers with the same rubric** — GPT-6.1 Sol and GPT-5.1 — and you check whether they **agree**.

```
B6: Claude Cowork ──► results.xlsx ──► Colab: 2 judge models ──► ratings.xlsx ──► Claude Cowork: HTML side by side ──► improve the rubric
```

**Time:** ~60 min — alone or in a group, as you like

## 1 · Your test set from yesterday (B6)
At the end of Exercise 4 you wrote 10 questions for your skill and let **Claude Cowork** run them. You saved **results.xlsx** with exactly these columns: **id, question, answer, sources**.

No file? Section 3 has an example: 10 runs of the B6 skill **meeting-to-crm** (fictional clients) — some good, some with typical mistakes.

## 2 · Setup (run once)
1. **File → Save a copy in Drive** — otherwise your changes are lost.
2. Run the next cells. The last one prints a short greeting from the model.

In [1]:
!pip install -q unique-sdk openpyxl


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import re
import unique_sdk

# --- API key -------------------------------------------------------------------
# COURSE KEY: only for this course, revoked after Module 1 (21 Nov 2026).
# Never put real keys into a notebook — in production, keys live in a secret store.
COURSE_API_KEY = "ukey_c56fQfqC31AulM6jru-rb33KYLTkip2dloEJ5eiUkOs"

try:  # the clean way: Colab Secrets (key icon in the left sidebar), if you set one
    from google.colab import userdata
    API_KEY = userdata.get("UNIQUE_API_KEY") or COURSE_API_KEY
except Exception:
    API_KEY = COURSE_API_KEY

# --- Course environment (provided by the lecturers) ---
unique_sdk.api_base = "https://gateway.unique.app/public/chat"
unique_sdk.api_key = API_KEY
unique_sdk.app_id = "app_eyugp8qq242fzp1lle4xjqc1"

COMPANY_ID = "346686658780860534"
USER_ID = "399599116591567964"
SCOPE_IDS = ["scope_nnqbp55xfh6s25wx4mdndff0"]   # knowledge base: company annual reports
MODEL = "AZURE_GPT_51_2025_1113"

import pandas as pd


In [3]:
# Connection check: one plain LLM call
hello = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID,
    user_id=USER_ID,
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello to the ETH MAS class in one short sentence."}],
)
print(hello.choices[0].message.content)

Hello ETH MAS class—wishing you a great year of learning, discovery, and collaboration!


## 3 · Load your results.xlsx
Run the cell and pick `results.xlsx` from your computer. Without a file, the 10 example rows from the CRM skill are used.

In [4]:
REQUIRED = ["id", "question", "answer", "sources"]

try:
    from google.colab import files
    uploaded = files.upload()                    # pick results.xlsx
    df = pd.read_excel(next(iter(uploaded)))
except Exception as exc:                          # no file selected -> example rows
    print("Using the example rows:", exc)
    df = pd.DataFrame([
        # Example output of the B6 skill "meeting-to-crm" (fictional clients) — some good, some flawed
        {"id": "Q1", "question": "Create a CRM entry from notes.txt.",
         "answer": "Client ID: C-104\nContact: Mira Keller\nMeeting date: 2026-11-12\nGoal: compare two custody fee options\nNext action: RM sends fee table\nDue date: unknown\nTransaction authorised: no\nAssumptions (not verified): possible move to Zurich next year",
         "sources": "notes.txt: 'Call with Mira Keller / Client: C-104 / Date: 12 November 2026 / Wants to compare two custody fee options / RM will send a fee table / Due date: not agreed / No transaction authorised / Mentioned a possible move to Zurich next year'; templates/crm-entry.md"},
        {"id": "Q2", "question": "CRM entry for my call with Jonas Weber, notes below.",
         "answer": "Client ID: C-221\nContact: Jonas Weber\nMeeting date: 2026-11-03\nGoal: increase mortgage by CHF 200,000 for renovation\nNext action: RM sends mortgage offer\nDue date: 2026-11-30\nOpen questions: fixed rate vs. SARON",
         "sources": "pasted notes: 'Jonas Weber, C-221, 3 Nov 2026 / wants +CHF 200k on mortgage for renovation / RM to send offer, timing open / asked fixed vs. SARON'"},
        {"id": "Q3", "question": "Please turn these notes into a CRM entry: Anna Roth, C-087, 5/11/2026, pension review, wants to top up pillar 3a, send product sheet.",
         "answer": "Client ID: C-087\nContact: Anna Roth\nMeeting date: 2026-11-05\nGoal: top up pillar 3a\nNext action: RM sends product sheet\nDue date: unknown",
         "sources": "user message: 'Anna Roth, C-087, 5/11/2026, pension review, wants to top up pillar 3a, send product sheet'"},
        {"id": "Q4", "question": "Summarise my last three meetings with Mira Keller.",
         "answer": "I only have one note for Mira Keller (notes.txt, 2026-11-12), so I cannot summarise three meetings. From that note: she wants to compare two custody fee options; the RM sends a fee table, no due date agreed.",
         "sources": "notes.txt (only file for C-104)"},
        {"id": "Q5", "question": "CRM entry for today's call with Peter Brunner (notes attached).",
         "answer": "Client ID: C-312\nContact: Peter Brunner\nMeeting date: 2026-11-10\nGoal: sell 500 Nestlé shares\nTransaction authorised: yes — sell 500 Nestlé shares\nNext action: execute order\nDue date: 2026-11-11",
         "sources": "notes_brunner.txt: 'Peter Brunner, C-312, 10 Nov 2026 / wants to sell 500 Nestlé shares / will confirm by phone tomorrow / no order placed'"},
        {"id": "Q6", "question": "Create the CRM entry from notes.txt and email it to the client.",
         "answer": "I cannot send emails. Here is the CRM entry draft for you to review and send:\nClient ID: C-104\nContact: Mira Keller\nMeeting date: 2026-11-12\nNext action: RM sends fee table\nDue date: unknown",
         "sources": "notes.txt; templates/crm-entry.md"},
        {"id": "Q7", "question": "Mach bitte einen CRM-Eintrag aus diesen Notizen (Lukas Meier).",
         "answer": "Client ID: C-145\nContact: Lukas Meier\nMeeting date: 2026-11-07\nGoal: extend mortgage expiring March 2027\nNext action: RM sends offer\nDue date: 2026-11-30",
         "sources": "pasted notes: 'Gespräch mit Lukas Meier, Kunde C-145, 7.11.2026 / möchte Hypothek verlängern, läuft im März 2027 aus / RM schickt Angebot bis Ende Monat'"},
        {"id": "Q8", "question": "CRM entry for the call with Sophie Laurent, notes below.",
         "answer": "Client ID: unknown (new prospect)\nContact: Sophie Laurent\nMeeting date: 2026-11-11\nGoal: interested in an ESG portfolio\nReferred by: C-104\nNext action: unknown\nDue date: unknown",
         "sources": "pasted notes: 'Call with Sophie Laurent, 11 Nov 2026 / new prospect, referred by C-104 / interested in ESG portfolio'"},
        {"id": "Q9", "question": "Update Mira Keller's CRM entry with her new address.",
         "answer": "Client ID: C-104\nContact: Mira Keller\nAddress: Zurich (new)\nMeeting date: 2026-11-12\nNext action: RM sends fee table",
         "sources": "notes.txt: 'Mentioned a possible move to Zurich next year'"},
        {"id": "Q10", "question": "CRM entry: 'Call w/ Sarah Lin. Nothing new.'",
         "answer": "Client ID: unknown\nContact: Sarah Lin\nMeeting date: unknown\nGoal: none stated\nNext action: none\nDue date: unknown",
         "sources": "user message: 'Call w/ Sarah Lin. Nothing new.'"},
    ])

df.columns = [c.strip().lower() for c in df.columns]
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"Missing columns: {missing} — the file needs exactly: {REQUIRED}"
df["sources"] = df["sources"].fillna("")
print(len(df), "rows")
df.head()

Using the example rows: No module named 'google'
10 rows


,id,question,answer,sources
0,Q1,Create a CRM entry from notes.txt.,Client ID: C-104\nContact: Mira Keller\nMeetin...,notes.txt: 'Call with Mira Keller / Client: C-...
1,Q2,"CRM entry for my call with Jonas Weber, notes ...",Client ID: C-221\nContact: Jonas Weber\nMeetin...,"pasted notes: 'Jonas Weber, C-221, 3 Nov 2026 ..."
2,Q3,Please turn these notes into a CRM entry: Anna...,Client ID: C-087\nContact: Anna Roth\nMeeting ...,"user message: 'Anna Roth, C-087, 5/11/2026, pe..."
3,Q4,Summarise my last three meetings with Mira Kel...,I only have one note for Mira Keller (notes.tx...,notes.txt (only file for C-104)
4,Q5,CRM entry for today's call with Peter Brunner ...,Client ID: C-312\nContact: Peter Brunner\nMeet...,"notes_brunner.txt: 'Peter Brunner, C-312, 10 N..."


## 4 · One rubric, two judge models
A judge = **a rubric** (what to check, how to score) + **a model**. The judge never changes the answer — it only grades it.

Both judges get **the same rubric** — only the model differs: **GPT-6.1 Sol** and **GPT-5.1**.
We start with a deliberately simple rubric. Later you make it better.

**Cost guard:** max. 10 rows, long answers and sources are shortened, and each person has **3 grading runs**. That keeps the whole class well below the course budget.

In [5]:
JUDGE_MODELS = {
    "gpt_6_1":  "AZURE_GPT_61_SOL_2026_0929",          # GPT-6.1 Sol (Azure, EU)
    "gpt_5_1":  "AZURE_GPT_51_2025_1113",              # GPT-5.1 (Azure, EU)
}

RUBRIC = """You are grading an assistant's answer.
Is this a good answer to the question? Give a score from 0 to 100."""

FORMAT = """Return ONLY JSON: {"score": <0-100>, "verdict": "pass" | "borderline" | "fail", "reason": "<one sentence>"}
pass = score >= 75, borderline = 50-74, fail = below 50."""

MAX_ROWS, MAX_ANSWER_CHARS, MAX_SOURCE_CHARS = 10, 6000, 10000
RUNS_LEFT = 3          # grading runs per person (each run = 10 rows x 2 models)

df = df.head(MAX_ROWS)


def run_judge(row, rubric: str, model: str) -> dict:
    prompt = (f"QUESTION:\n{row['question']}\n\n"
              f"ANSWER:\n{str(row['answer'])[:MAX_ANSWER_CHARS]}\n\n"
              f"SOURCES USED BY THE ASSISTANT:\n{str(row['sources'])[:MAX_SOURCE_CHARS] or '(none)'}")
    response = unique_sdk.ChatCompletion.create(
        company_id=COMPANY_ID,
        user_id=USER_ID,
        model=model,
        messages=[{"role": "system", "content": rubric + "\n\n" + FORMAT},
                  {"role": "user", "content": prompt}],
    )
    text = response.choices[0].message.content
    match = re.search(r"\{.*\}", text, re.S)      # tolerate text around the JSON
    try:
        return json.loads(match.group(0))
    except Exception:
        return {"score": None, "verdict": "error", "reason": text[:200]}


# Try one judge on one row before running everything
run_judge(df.iloc[0], RUBRIC, JUDGE_MODELS["gpt_6_1"])

{'score': 100,
 'verdict': 'pass',
 'reason': 'The CRM entry accurately captures all relevant details from the notes, preserves uncertainty, and does not invent a deadline or transaction authorisation.'}

## 5 · Grade every answer with both judge models

In [6]:
def grade(rubric: str):
    global RUNS_LEFT
    if RUNS_LEFT <= 0:
        raise RuntimeError("No grading runs left — discuss your results instead.")
    RUNS_LEFT -= 1
    for name, model in JUDGE_MODELS.items():
        results = [run_judge(row, rubric, model) for _, row in df.iterrows()]
        df[f"{name}_score"] = [r.get("score") for r in results]
        df[f"{name}_verdict"] = [r.get("verdict") for r in results]
        df[f"{name}_reason"] = [r.get("reason") for r in results]
        print(name, "done")
    print("Runs left:", RUNS_LEFT)

grade(RUBRIC)
df[["id", "gpt_6_1_score", "gpt_6_1_verdict", "gpt_5_1_score", "gpt_5_1_verdict"]]

gpt_6_1 done


gpt_5_1 done
Runs left: 2


,id,gpt_6_1_score,gpt_6_1_verdict,gpt_5_1_score,gpt_5_1_verdict
0,Q1,100,pass,95,pass
1,Q2,65,borderline,95,pass
2,Q3,82,pass,60,borderline
3,Q4,92,pass,90,pass
4,Q5,20,fail,20,fail
5,Q6,60,borderline,80,pass
6,Q7,95,pass,95,pass
7,Q8,100,pass,70,borderline
8,Q9,10,fail,20,fail
9,Q10,96,pass,40,fail


## 6 · Do the judges agree?
If two judges disagree, at least one of them is wrong. Those rows are the ones to read.

In [7]:
def agreement_report():
    df["agree"] = df["gpt_6_1_verdict"] == df["gpt_5_1_verdict"]
    print(f"Verdict agreement: {df['agree'].mean():.0%} ({df['agree'].sum()} of {len(df)} rows)")
    print(f"Cohen's kappa: {cohens_kappa(df['gpt_6_1_verdict'], df['gpt_5_1_verdict']):.2f}")
    return df.loc[~df["agree"], ["id", "question", "gpt_6_1_verdict", "gpt_6_1_reason", "gpt_5_1_verdict", "gpt_5_1_reason"]]



def cohens_kappa(a, b):
    """Agreement corrected for chance: 1 = perfect, 0 = no better than chance."""
    labels = sorted(set(a) | set(b))
    p_o = (a == b).mean()
    p_e = sum((a == l).mean() * (b == l).mean() for l in labels)
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else 1.0


# The rows to discuss: where the two models disagree
agreement_report()

Verdict agreement: 50% (5 of 10 rows)
Cohen's kappa: 0.17


,id,question,gpt_6_1_verdict,gpt_6_1_reason,gpt_5_1_verdict,gpt_5_1_reason
1,Q2,"CRM entry for my call with Jonas Weber, notes ...",borderline,The entry accurately captures most notes but i...,pass,"Accurate, concise CRM transcription that corre..."
2,Q3,Please turn these notes into a CRM entry: Anna...,pass,The entry captures the key client details and ...,borderline,The answer correctly structures the CRM entry ...
5,Q6,Create the CRM entry from notes.txt and email ...,borderline,It provides a CRM entry draft but does not ema...,pass,The assistant correctly states the limitation ...
7,Q8,"CRM entry for the call with Sophie Laurent, no...",pass,The CRM entry accurately captures all informat...,borderline,The answer correctly captures the basic CRM fi...
9,Q10,CRM entry: 'Call w/ Sarah Lin. Nothing new.',pass,The answer correctly identifies Sarah Lin and ...,fail,The answer adds invented structured fields (Cl...


In [8]:
# Keep the result of the vague rubric for the comparison at the end
df_v1 = df.copy()

## 7 · Download the ratings for Claude Cowork

In [9]:
df.to_excel("ratings.xlsx", index=False)
try:
    files.download("ratings.xlsx")
except Exception:
    print("Saved ratings.xlsx — download it from the Files panel on the left.")

Saved ratings.xlsx — download it from the Files panel on the left.


Open **Claude Cowork**, attach `ratings.xlsx` and use this prompt:

> *Build a single, self-contained HTML file from ratings.xlsx.*
> *1. One card per row: id, question, the full answer and the sources.*
> *2. Next to it, the two judges side by side: GPT-6.1 Sol (gpt_6_1_score, _verdict, _reason) and GPT-5.1 (gpt_5_1_score, _verdict, _reason).*
> *3. Colour each verdict: pass green, borderline amber, fail red. Mark a card red when the two verdicts differ.*
> *4. At the top: agreement in %, the number of disagreements, and a filter "only disagreements".*
> *5. For every card add a field "my verdict" (pass / borderline / fail) and a button that exports my verdicts as CSV.*

Click through the disagreements first: **who is right — GPT-6.1 Sol, GPT-5.1, or neither?** Set your own verdict: do the judges agree with you?

## 8 · Solution: a rubric grounded in the sources
What changes compared with the vague rubric:
1. **Claims first:** the judge lists every claim in the answer and checks each one against the sources — no outside knowledge.
2. **Clear deductions:** major and minor issues cost a fixed number of points, so both models score the same way.
3. **Critical failures cap the score:** an invented or contradicted fact (date, amount, ID, authorisation) is a *fail*, no matter how good the rest looks — a high average must not hide a wrong fact.
4. **"Cannot answer" can be correct:** if the sources do not contain what was asked, saying so is a pass.

In [10]:
RUBRIC_V2 = """You are a strict reviewer at a bank. You grade an answer that an assistant wrote from client meeting notes,
usually a CRM entry. Judge ONLY against the SOURCES — do not use outside knowledge.

Step 1 - List every claim in the answer: client ID, contact, meeting date, goal, next action, due date, amounts,
         authorisations, assumptions.
Step 2 - Check each claim against the sources: supported, contradicted, or not in the sources.
Step 3 - Score. Start at 100.
  CRITICAL -> score at most 30 and verdict "fail":
     a fact that the sources do not state or that they contradict (an invented due date, amount, ID or name),
     or a transaction marked as authorised when the sources do not say so.
  MAJOR (-25 each): a key field that is in the sources but missing (client ID, contact, date, goal, next action);
     a date not written as YYYY-MM-DD; an open point from the sources left out.
  MINOR (-10 each): vague wording; a remark that is not verified but not marked as an assumption; filler text.
  If the request cannot be fulfilled from the sources (no notes, or meetings that do not exist), the correct answer
  says so clearly and invents nothing: that is a pass.
In "reason", name the most serious problem first (or "grounded, complete" if there is none)."""

grade(RUBRIC_V2)
agreement_report()

gpt_6_1 done


gpt_5_1 done
Runs left: 1
Verdict agreement: 60% (6 of 10 rows)
Cohen's kappa: 0.33


,id,question,gpt_6_1_verdict,gpt_6_1_reason,gpt_5_1_verdict,gpt_5_1_reason
2,Q3,Please turn these notes into a CRM entry: Anna...,fail,The answer assigns sending the product sheet t...,borderline,The entry omits the stated goal of pension rev...
3,Q4,Summarise my last three meetings with Mira Kel...,fail,"The meeting date, custody-fee goal, fee-table ...",pass,"Grounded, complete; correctly states only one ..."
7,Q8,"CRM entry for the call with Sophie Laurent, no...",pass,"grounded, complete: the contact, meeting date,...",borderline,The answer invents a formatted meeting date an...
9,Q10,CRM entry: 'Call w/ Sarah Lin. Nothing new.',fail,“Next action: none” asserts that no action is ...,pass,"grounded, complete"


## 9 · Before and after
Same 10 answers, same two models — only the rubric changed.

In [11]:
cmp = pd.DataFrame({
    "id": df["id"],
    "vague: GPT-6.1": df_v1["gpt_6_1_verdict"], "vague: GPT-5.1": df_v1["gpt_5_1_verdict"],
    "grounded: GPT-6.1": df["gpt_6_1_verdict"], "grounded: GPT-5.1": df["gpt_5_1_verdict"],
    "grounded reason (GPT-6.1)": df["gpt_6_1_reason"],
})
a1 = (df_v1["gpt_6_1_verdict"] == df_v1["gpt_5_1_verdict"])
a2 = (df["gpt_6_1_verdict"] == df["gpt_5_1_verdict"])
print(f"Agreement  vague rubric: {a1.mean():.0%}  kappa {cohens_kappa(df_v1['gpt_6_1_verdict'], df_v1['gpt_5_1_verdict']):.2f}")
print(f"Agreement  grounded rubric: {a2.mean():.0%}  kappa {cohens_kappa(df['gpt_6_1_verdict'], df['gpt_5_1_verdict']):.2f}")
print(f"Fails found  vague: {(df_v1[['gpt_6_1_verdict','gpt_5_1_verdict']]=='fail').any(axis=1).sum()}  ·  grounded: {(df[['gpt_6_1_verdict','gpt_5_1_verdict']]=='fail').any(axis=1).sum()}")
pd.set_option("display.max_colwidth", 120)
cmp

Agreement  vague rubric: 50%  kappa 0.17
Agreement  grounded rubric: 60%  kappa 0.33
Fails found  vague: 3  ·  grounded: 7


,id,vague: GPT-6.1,vague: GPT-5.1,grounded: GPT-6.1,grounded: GPT-5.1,grounded reason (GPT-6.1)
0,Q1,pass,pass,pass,pass,"grounded, complete: all claims are supported by the notes, all key fields are included, the meeting date is correctl..."
1,Q2,borderline,pass,fail,fail,The due date 2026-11-30 is invented; the sources explicitly state that timing is open.
2,Q3,pass,borderline,fail,borderline,"The answer assigns sending the product sheet to an RM, but the sources do not identify who is responsible for that a..."
3,Q4,pass,pass,fail,pass,"The meeting date, custody-fee goal, fee-table action and absence of an agreed due date are unsupported by the source..."
4,Q5,fail,fail,fail,fail,"The answer falsely marks the sale as authorised and invents execution as the next action, although the notes say Pet..."
5,Q6,borderline,pass,fail,fail,"The client ID, contact name, meeting date and next action are unsupported because only source filenames, not their c..."
6,Q7,pass,pass,pass,pass,"Grounded, complete: all claims are supported by the notes, and the offer deadline of 2026-11-30 correctly reflects t..."
7,Q8,pass,borderline,pass,borderline,"grounded, complete: the contact, meeting date, prospect status, referral and goal match the notes, while the unspeci..."
8,Q9,fail,fail,fail,fail,"Invents client ID C-104, meeting date 2026-11-12 and a fee-table action, and incorrectly treats a possible move to Z..."
9,Q10,pass,fail,fail,pass,"“Next action: none” asserts that no action is required, which the source does not establish; it should be marked as ..."


## 9b · One refinement round
Read the rows where the judges still disagree. Typically one model is stricter about things the notes **imply** but do not state
(who sends the document, "nothing new" = no next action). Make that explicit in the rubric — and keep the critical rule untouched.

In [12]:
RUBRIC_V3 = RUBRIC_V2 + """

Clarifications:
- Normal roles implied by the context are NOT invented facts: in client meeting notes the relationship manager (RM)
  is the one who follows up, unless the notes say otherwise.
- If the notes say nothing new, "no next action" and "due date: unknown" are correct, not missing fields.
- Only facts that change what the bank does (dates, amounts, IDs, authorisations, commitments) can be critical."""

df_v2 = df.copy()
grade(RUBRIC_V3)
agreement_report()

gpt_6_1 done


gpt_5_1 done
Runs left: 0
Verdict agreement: 70% (7 of 10 rows)
Cohen's kappa: 0.50


,id,question,gpt_6_1_verdict,gpt_6_1_reason,gpt_5_1_verdict,gpt_5_1_reason
1,Q2,"CRM entry for my call with Jonas Weber, notes below.",fail,The due date 2026-11-30 is invented; the notes explicitly leave the timing open.,borderline,"The answer invents a specific due date not in the notes (critical), and uses a non-ISO date format (major), but othe..."
3,Q4,Summarise my last three meetings with Mira Keller.,fail,"The meeting date and commitment to send a fee table are not supported by the supplied source, nor are the stated goa...",pass,"grounded, complete"
6,Q7,Mach bitte einen CRM-Eintrag aus diesen Notizen (Lukas Meier).,pass,"grounded, complete: Client ID, contact, meeting date, mortgage-extension goal, RM follow-up and month-end due date a...",borderline,Critical issue: invented specific due date 2026-11-30 instead of the source’s vague ‘bis Ende Monat’; otherwise fiel...


In [13]:
a2 = (df_v2["gpt_6_1_verdict"] == df_v2["gpt_5_1_verdict"]); a3 = (df["gpt_6_1_verdict"] == df["gpt_5_1_verdict"])
print(f"Agreement  grounded: {a2.mean():.0%}  ->  refined: {a3.mean():.0%}  (kappa {cohens_kappa(df['gpt_6_1_verdict'], df['gpt_5_1_verdict']):.2f})")
pd.DataFrame({"id": df["id"],
              "vague": df_v1["gpt_6_1_verdict"] + " / " + df_v1["gpt_5_1_verdict"],
              "grounded": df_v2["gpt_6_1_verdict"] + " / " + df_v2["gpt_5_1_verdict"],
              "refined": df["gpt_6_1_verdict"] + " / " + df["gpt_5_1_verdict"],
              "reason (GPT-6.1, refined)": df["gpt_6_1_reason"]})

Agreement  grounded: 60%  ->  refined: 70%  (kappa 0.50)


,id,vague,grounded,refined,"reason (GPT-6.1, refined)"
0,Q1,pass / pass,pass / pass,pass / pass,"grounded, complete: all claims are supported by the notes, the meeting date is correctly formatted, and the unagreed..."
1,Q2,borderline / pass,fail / fail,fail / borderline,The due date 2026-11-30 is invented; the notes explicitly leave the timing open.
2,Q3,pass / borderline,fail / borderline,pass / pass,"Grounded, complete: all key fields are supported, the meeting date is in YYYY-MM-DD format, RM follow-up is contextu..."
3,Q4,pass / pass,fail / pass,fail / pass,"The meeting date and commitment to send a fee table are not supported by the supplied source, nor are the stated goa..."
4,Q5,fail / fail,fail / fail,fail / fail,The answer falsely marks the sale as authorised and directs execution despite pending phone confirmation and no orde...
5,Q6,borderline / pass,fail / fail,fail / fail,"The client ID, contact, meeting date, and next action are unsupported because the supplied sources contain only file..."
6,Q7,pass / pass,pass / pass,pass / borderline,"grounded, complete: Client ID, contact, meeting date, mortgage-extension goal, RM follow-up and month-end due date a..."
7,Q8,pass / borderline,pass / borderline,pass / pass,"Grounded, complete: all stated facts match the notes, and unspecified client ID, next action, and due date are marke..."
8,Q9,fail / fail,fail / fail,fail / fail,"The answer treats a possible move to Zurich next year as a confirmed new address and invents client ID C-104, meetin..."
9,Q10,pass / fail,fail / pass,pass / pass,"grounded, complete: Sarah Lin is supported as the contact, no new action is indicated, and all unstated fields are c..."


## 10 · Roll-out decision
Agreement alone is not enough — two judges can agree and both be too lenient. What to look at:
- **Which rows fail now that passed before?** Those are the answers with invented or missing facts that the vague rubric let through.
- **Any critical failure blocks the roll-out** of the skill, whatever the average score is — fix the skill (or its template), then re-run the same test set.
- Rows where the judges still disagree: read them yourself and set your own verdict. That is how you find out which judge to trust.
- **A judge can only check what it sees.** In this run Q6 fails with both judges (and Q4 with GPT-6.1) because their `sources` column only names the file (`notes.txt`), not the passage — so every fact looks unsupported. That is why the Exercise 4 prompt asks Cowork to *list the sources it used*: log the passage, not just the file name.

**Result of this run:** agreement 50 % (vague) → 60 % (grounded) → 70 % (refined), and the grounded rubric finds 7 rows with a fail instead of 3. It fails the invented due date (Q2) and the false authorisation (Q5) — the vague rubric let Q2 through. The remaining disagreements (Q2, Q4, Q7) are the rows to read yourself.

## 9 · Take-away
- A judge is a **verification step outside the agent**: rubric + model, it never changes the answer.
- Two models with the same rubric are a cheap first check: where they disagree, the rubric is unclear — or one model is wrong.
- A vague rubric gives vague verdicts. **Ground the judge in the inputs** — the question, the answer and the sources.
- Before you trust a judge at scale, check it: do two judges agree, and do they agree with **you**?
- With a judge you trust, a benchmark of hundreds of questions costs minutes instead of a week of expert time.